# 🚀 Detección de Fraude en Tarjetas de Crédito

Este notebook guía el proceso completo para entrenar, validar y evaluar modelos de machine learning en la detección de transacciones fraudulentas. Se abordan técnicas de validación cruzada, comparación de algoritmos clásicos y avanzados, y análisis detallado de resultados sobre datos reales. 🔍💳

> **Objetivo:** Identificar patrones y señales relevantes en las transacciones para construir modelos robustos que ayuden a prevenir el fraude financiero.

In [1]:
# librerías
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import  cross_val_score, ShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

📦 **Carga de Datos Procesados**

En esta sección se cargan los conjuntos de datos ya procesados y listos para el entrenamiento y evaluación de los modelos. Esto permite trabajar directamente con las variables relevantes y acelerar el flujo de trabajo en la detección de fraude. ⚡️

In [24]:
# Cargar datos procesados
X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f'train shape:{X_train.shape}, test shape:{X_test.shape}')
print(f'Fraude en train: {np.mean(y_train):.4f}, Fraude en test: {np.mean(y_test):.4f}')

column_names = pd.read_csv('../data/raw/creditcard.csv', nrows=1).columns.tolist()
X_test = pd.DataFrame(X_test, columns=column_names[:-1])


train shape:(199364, 30), test shape:(85443, 30)
Fraude en train: 0.0017, Fraude en test: 0.0017


🔄 **Definición de Validación Cruzada**

En esta sección se establece la estrategia de validación cruzada, fundamental para evaluar el desempeño de los modelos en conjuntos de datos desbalanceados. Utilizamos técnicas robustas como *StratifiedKFold* para asegurar que cada partición mantenga la proporción de clases, permitiendo una comparación justa y confiable entre algoritmos. 🧪📊

In [5]:
from sklearn.model_selection import StratifiedKFold

# Estrategia robusta para datasets desbalanceados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

⚡️ **Modelos Baseline**

En esta sección se evalúan los modelos clásicos de machine learning como referencia inicial (*baseline*) para la detección de fraude. Estos algoritmos permiten establecer un punto de comparación y entender el desempeño mínimo esperado antes de aplicar técnicas más avanzadas. 🏁🔍

In [12]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=200, class_weight="balanced", random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=42)
}

cv_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # Menos splits

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv_fast, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f'{name}: AUC ROC mean={scores.mean():.4f}, std={scores.std():.4f}')

LogisticRegression: AUC ROC mean=0.9808, std=0.0046
RandomForest: AUC ROC mean=0.9411, std=0.0019


🌟 **Modelos Avanzados**

En esta sección se exploran algoritmos de machine learning más sofisticados, como XGBoost y LightGBM, que suelen ofrecer mejor desempeño en la detección de fraude gracias a su capacidad para capturar patrones complejos y manejar grandes volúmenes de datos. Estos modelos permiten optimizar la precisión y robustez de las predicciones, superando los enfoques clásicos. 🚦🤖

In [13]:
models_adv={
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

for name, model in models_adv.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f'{name}: AUC ROC mean={scores.mean():.4f}, std={scores.std():.4f}')

XGBoost: AUC ROC mean=0.9159, std=0.0212
LightGBM: AUC ROC mean=0.6958, std=0.2254


🏆 **Entrenamiento del Modelo Final**

En esta etapa se selecciona y entrena el mejor modelo identificado durante la validación, optimizando sus parámetros para maximizar la detección de fraude. El modelo final se guarda para su uso en producción y futuras evaluaciones. 🚨🤖

In [14]:
# Supongamos que LightGBM fue el mejor
final_model = LGBMClassifier(class_weight="balanced", random_state=42)
final_model.fit(X_train, y_train)

# Guardar modelo
joblib.dump(final_model, '../models/lgbm_fraud_model.pkl')
print('Modelo guardado en ../models/lgbm_fraud_model.pkl')

[LightGBM] [Info] Number of positive: 344, number of negative: 199020
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035246 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7420
[LightGBM] [Info] Number of data points in the train set: 199364, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Modelo guardado en ../models/lgbm_fraud_model.pkl


🚦 **Evaluación Rápida en Test**

En esta sección se realiza una evaluación ágil del modelo final sobre el conjunto de test, permitiendo visualizar métricas clave como la matriz de confusión, el reporte de clasificación y el AUC ROC. Así comprobamos el poder predictivo del modelo ante datos no vistos y su capacidad para detectar fraudes reales. 📊🔍

In [25]:
# Evaluación en el conjunto de test
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(f'ROC AUC: {roc_auc_score(y_test, y_prob):.4f}')

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.83      0.78      0.80       148

    accuracy                           1.00     85443
   macro avg       0.92      0.89      0.90     85443
weighted avg       1.00      1.00      1.00     85443

Confusion Matrix:
[[85272    23]
 [   33   115]]
ROC AUC: 0.9451
